# Bussin -- Colab free worker

Colab free is pre-emptible and gives roughly 4 h sessions, so the watchdog uses a shorter budget here. It contributes real hours but should not be the primary carrier of a run.

Add `HF_TOKEN` via the key icon in the left sidebar.


In [ ]:
# --- Bussin worker on Colab free --------------------------------------
import os, subprocess, sys, time
T0 = time.time()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if not os.path.exists("bussin"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/CHANGEME/bussin.git", "/content/bussin"],
                   check=False)
os.chdir("/content/bussin")
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "hf_transfer", "safetensors", "pyyaml", "regex"], check=False)

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print(f"setup took {time.time() - T0:.1f}s")

In [ ]:
CONFIG = "configs/400m.yaml"

In [ ]:
# --- Corpus ------------------------------------------------------------
# Colab cannot mount a Kaggle Dataset, so it pulls the shards it needs from the
# HF mirror. Only fetch what this session will actually consume: the free tier
# is pre-emptible and a 60 GB download would never finish.
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="CHANGEME/bussin-corpus", repo_type="dataset",
    allow_patterns=["manifest.json", "shards/part-00/*"],
    local_dir="/content/corpus",
)

In [ ]:
# --- What did we wake up on? -----------------------------------------
from bussin.relay.platform import get_platform_info, plan_batch

info = get_platform_info()
print(info)

# The bf16 question decides the whole precision path. Kaggle's T4 is Turing
# (sm75) and the P100 is Pascal (sm60); bf16 tensor cores start at Ampere, so
# on Kaggle GPUs this always prints False and training runs fp16 + GradScaler.
print(f"bf16 available: {info.supports_bf16}")
if info.device_type == "cuda" and "P100" in info.device_name:
    print("WARNING: P100 selected. It has no tensor cores and is roughly 6x "
          "slower than T4 x2 for the same quota hour. Switch to T4 x2.")

In [ ]:
# --- Train ------------------------------------------------------------
# The worker claims the lease, restores the run, trains until the watchdog
# fires at (session limit - 20 min), checkpoints, releases the lease, exits.
#
# If another worker holds a live lease this exits in seconds without starting,
# so a scheduled notebook costs nothing when the run is already being carried.
from bussin.relay.bootstrap import run

exit_code = run(CONFIG)
print(f"worker exited {exit_code}")